# Word Count on Amazon Reviews

In [1]:
from pyspark.sql import SparkSession

# SparkSession is the single entry point to everything in Spark.
# Think of it as the "connection" to your Spark engine,
# the same way you'd connect to a database before running queries.

spark = (
    SparkSession
    .builder
    .appName("WordCount")   # A label for your job — visible in the Spark UI at localhost:4040
    .master("local[*]")     # Run locally using ALL available CPU cores.
                            # On a real cluster this would be "spark://host:7077" or "yarn".
    .getOrCreate()          # Create a new session, or reuse one if it already exists.
)

# sc = SparkContext. This is the older, lower-level object that SparkSession wraps.
# We need it directly here because RDD operations live on sc, not spark.
sc = spark.sparkContext

In [2]:
# RDD is one of the core objects of PySpark like DataFrame object for pandas.
reviews = sc.textFile('amazon_cells_labelled.txt')
type(reviews)  # Loaded as an RDD object. RDD - Resilient Distributed Dataset

pyspark.core.rdd.RDD

In [3]:
reviews.take(5)  # Sample few lines of reviews

['So there is no way for me to plug it in here in the US unless I go by a converter.\t0',
 'Good case, Excellent value.\t1',
 'Great for the jawbone.\t1',
 'Tied to charger for conversations lasting more than 45 minutes.MAJOR PROBLEMS!!\t0',
 'The mic is great.\t1']

In [4]:
# Split the sentences into words for counting purpose.
words = reviews.flatMap(lambda review: review.split(" "))

In [5]:
words  # No output is produced because flatMap() function is a transformation (lazily evaluated)

PythonRDD[3] at RDD at PythonRDD.scala:57

In [6]:
words.count()  # count() is an action. Eagerly evaluated. There are 10246 words in the reviews.

10246

In [10]:
words.take(25) # flatMap() splits the sentences into words based on the function given.

['So',
 'there',
 'is',
 'no',
 'way',
 'for',
 'me',
 'to',
 'plug',
 'it',
 'in',
 'here',
 'in',
 'the',
 'US',
 'unless',
 'I',
 'go',
 'by',
 'a',
 'converter.\t0',
 'Good',
 'case,',
 'Excellent',
 'value.\t1']

In [13]:
# Remove numbers, punctuations, special characters.
# First replace the above with empty string.
import re
words_processed = words.map(lambda word: re.sub(r'[^A-Za-z]', '', word))
words_processed.take(25)

['So',
 'there',
 'is',
 'no',
 'way',
 'for',
 'me',
 'to',
 'plug',
 'it',
 'in',
 'here',
 'in',
 'the',
 'US',
 'unless',
 'I',
 'go',
 'by',
 'a',
 'converter',
 'Good',
 'case',
 'Excellent',
 'value']

In [14]:
# Filter out the empty strings
words_clean = words_processed.filter(lambda word: word != "")
words_clean.count()  # Some words have been removed because they don't contain alphabets.

10118

In [16]:
# Remove stop words
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [18]:
from nltk.corpus import stopwords

stop_words = stopwords.words('english')

In [19]:
# Filter out stop words
words_filtered = words_clean.filter(lambda word: word not in stop_words)
words_filtered.count()  # The total word count reduced by almost half after removing stop words

6060

In [20]:
# Map each word to have count 1
word_initial_count = words_filtered.map(lambda word: (word, 1))
word_initial_count.take(10)

[('So', 1),
 ('way', 1),
 ('plug', 1),
 ('US', 1),
 ('unless', 1),
 ('I', 1),
 ('go', 1),
 ('converter', 1),
 ('Good', 1),
 ('case', 1)]

In [21]:
# Count the total occurrences of each word
# For example, reduceByKey() works as follows:
# Groups by keys first: So, ('great', 1), ('great', 1), ('great', 1)
# Then apply the lambda function. ('great', 1) and ('great', 1) is taken and only values are added.
# Next step: ('great', 2) and ('great', 1) are taken. Then, ('great', 3) becomes the result.
word_total_count = word_initial_count.reduceByKey(lambda count1, count2: count1 + count2)
word_total_count.take(5)

[('unless', 2), ('I', 306), ('converter', 1), ('Good', 14), ('Excellent', 14)]

In [26]:
# Sort in descending order of occurrence of words
sorted_word_total_count = word_total_count.sortBy(lambda pair: pair[1], ascending=False)
sorted_word_total_count.take(5)

[('I', 306), ('phone', 148), ('The', 97), ('great', 66), ('good', 63)]